In [1]:

import pandas as pd
import oracledb 
import datetime

# --- 1. CONFIGURATION ---

# Connection details (Placeholders for USER, PASSWORD, HOST, etc.)
HOST = "157.0.2.23"
PORT = 1521
SERVICE_NAME = "imdb"
USER = "imonitor"
PASSWORD = "imtg" # Use your actual password

# Path to the Oracle Instant Client (if required for thick mode)
ORACLE_CLIENT_LIB_DIR = "D:/app/oracle11gx64/product/11.2.0/client_1/BIN" 

# Time window (Use datetime objects for robust binding)
try:
    START_DATE = datetime.datetime.strptime('2025-04-01', '%Y-%m-%d')
    END_DATE = datetime.datetime.strptime('2026-02-06', '%Y-%m-%d')
except ValueError as e:
    print(f"Error parsing dates: {e}")
    START_DATE = '2025-04-01'
    END_DATE = '2025-11-26'


# --- 2. MODIFIED SQL QUERY (NOW INCLUDES KILN FILTER) ---

# We add: AND T_TSBSL_DRI_PROD_DAILY.SOURCE = :kiln_filter
# Ensure :kiln_filter is included in the bind_vars below.

SQL_QUERY = """
SELECT *
FROM TSBSL.T_TSBSL_DRI_PROD_DAILY T_TSBSL_DRI_PROD_DAILY
WHERE (T_TSBSL_DRI_PROD_DAILY.TIMESTAMP >= :start_date AND T_TSBSL_DRI_PROD_DAILY.TIMESTAMP <= :end_date)
  AND T_TSBSL_DRI_PROD_DAILY.SOURCE = :kiln_filter
ORDER BY T_TSBSL_DRI_PROD_DAILY.TIMESTAMP
"""

# --- 3. FUNCTION TO CONNECT AND FETCH DATA ---

def fetch_oracle_data_to_dataframe():
    
    connection = None
    data_frame = pd.DataFrame()
    
    try:
        # Initialize oracledb (Thick Mode)
        if ORACLE_CLIENT_LIB_DIR:
             oracledb.init_oracle_client(lib_dir=ORACLE_CLIENT_LIB_DIR)
             print("Initialized Oracle Client (Thick Mode).")
             
        dsn = oracledb.makedsn(HOST, PORT, service_name=SERVICE_NAME)
        
        # Establish the connection
        connection = oracledb.connect(user=USER, password=PASSWORD, dsn=dsn)
        print("Successfully connected to Oracle database.")
        
        # Define the parameter dictionary (NOW INCLUDES KILN FILTER)
        bind_vars = {
            'start_date': START_DATE,
            'end_date': END_DATE,
            'kiln_filter': 'KILN1' # The filter value for Kiln 1
        }
        
        # Execute the query and load into DataFrame
        data_frame = pd.read_sql(SQL_QUERY, connection, params=bind_vars)
        
        print("Data successfully loaded into Pandas DataFrame.")
        print(f"Total rows fetched for KILN1: {len(data_frame)}")
        
    except oracledb.Error as e:
        error, = e.args
        print(f"Database Error: ORA-{error.code}: {error.message}")
        print("Please verify the SQL query, connection details, and table/column names.")
        
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        
    finally:
        # Close the connection
        if connection:
            connection.close()
            print("Connection closed.")
            
    return data_frame

# --- 4. EXECUTION ---
df_coal = fetch_oracle_data_to_dataframe()

# Display the head of the DataFrame
if not df_coal.empty:
    print("\n--- Extracted Data (Head) ---")
    print(df_coal.head())

ModuleNotFoundError: No module named 'oracledb'

In [ ]:
import oracledb
import pandas as pd

# -------- CONFIG --------
START_DATE = "2025-04-01 00:00"
END_DATE   = "2025-11-26 23:59"

# Oracle Client init (safe even if called earlier)
try:
    oracledb.init_oracle_client(
        lib_dir=r"D:/app/Oracle11gX64/product/11.2.0/client_1/BIN"
    )
except oracledb.ProgrammingError:
    # Already initialized in this process
    pass

dsn = oracledb.makedsn("157.0.2.23", 1521, service_name="imdb")

# -------- COLUMN LIST (from your DAILY query) --------
COLUMNS = [
    "TIMESTAMP",
    "ZONE1_TEMP", "ZONE2_TEMP", "ZONE3_TEMP", "ZONE4_TEMP", "ZONE5_TEMP",
    "ZONE6_TEMP", "ZONE7_TEMP", "ZONE8_TEMP", "ZONE9_TEMP", "ZONE10_TEMP",
    "ZONE11_TEMP",
    "IRON_ORE_FEED_RATE", "FD_COAL_FEED_RATE", "DOLO_FEED_RATE",
    "FINES_COAL_FEED_RATE", "CC_FEED_RATE", "FEED_MIX_FEED_RATE", "KMD_RPM",
    "CHAR_FEED_RATE", "FINES_FEED_RATE", "LUMPS_FEED_RATE",
    "IRON_ORE_FEED_RATE_PLUS_STBY", "FEED_COAL_FR_MAIN_PLUS_STBY",
    "ELBOW_DUCT_TEMP", "INLET_HOOD_TEMP", "OUTLET_HOOD_TEMP", "STEAM_FLOW",
    "CD_TEMPERATURE", "WHRB_DAMPER_POSITION", "BOILER_INLET_TEMP",
    "WHRB_DUST_EMMISION",
    "UPPER_ABC_ZONE3_TEMP", "UPPER_ABC_ZONE4_TEMP", "UPPER_ABC_ZONE5_TEMP",
    "LOWER_ABC_ZONE6_TEMP", "LOWER_ABC_ZONE7_TEMP", "LOWER_ABC_ZONE8_TEMP",
    "DSC_BACKFLOW_TEMP", "DSC_ZONE10_TEMP", "DSC_ZONE11_TEMP",
    "LOBE_PRESSURE", "KILN_INLET_PRESSURE", "KILN_OUTLET_PRESSURE",
    "COOLER_INLET_PRESSURE", "COOLER_OUTLET_PRESSURE",
    "SAF_FAN_SET_POINT", "KMD_RPM_SET_POINT", "FINES_COAL_SET_POINT",
    "COARSE_COAL_SET_POINT", "IRON_ORE_SET_POINT_STBY", "IRON_ORE_SET_POINT",
    "FEED_COAL_SET_POINT_STBY", "FEED_COAL_SET_POINT", "DOLOMITE_SET_POINT",
    "CB_FAN_SET_POINT", "OVER_SIZE_SET", "FINES_SET", "LUMPS_SET",
    "KMD_CURRENT_MASTER", "KMD_CURRENT_SLAVE", "LOBE_FLOW", "CB_AIR_FLOW",
    "CB__CURRENT", "SAF_CURRENT", "CD_BELT_SPEED", "CD_HOPPER_LEVEL",
    "CD_MATERIAL_FLOW", "COOLER_WATER_FLOW",
    "U_ABC_AIR_FLOW", "M_ABC_AIR_FLOW",
    "U_ABC_CURRENT", "M_ABC_CURRENT", "L_ABC_CURRENT", "CMD_CURRENT",
    "KMD_MASTER_RTD1", "KMD_MASTER_RTD2", "KMD_SLAVE_RTD1",
    "KMD_SLAVE_RTD2",
    "WHRB_CURRENT",
    "IRON_ORE_BUNKER_LEVEL", "FD_COAL_BUNKER_LEVEL", "DOLO_BUNKER_LEVEL",
    "FINES_COAL_BUNKER_LEVEL", "CC_BUNKER_LEVEL",
    "WHRB_O2", "WHRB_CO2", "WHRB_CO_PPM", "WHRB_NO_PPM", "WHRB_NO2_PPM",
    "WHRB_NOX_PPM", "WHRB_SO2_PPM",
    "COMMON_BC_15_FLOW", "COMMON_DRI_FINES_FLOW", "COMMON_DRI_CHAR_FLOW",
    "COMMON_DRI_LUMP_FLOW",
    "SAF_1_SECONDARY_AIR", "SAF_2__SECONDARY_AIR", "SAF_3_SECONDARY_AIR",
    "SAF_4_SECONDARY_AIR", "SAF_5_SECONDARY_AIR", "SAF_6_SECONDARY_AIR",
    "SAF_7_SECONDARY_AIR", "SAF_8_SECONDARY_AIR",
    "COM_SILO_LEVEL", "COM_SILO_BAG_FILTER_DPT", "COM_SINGLE_DRUM_SPEED",
    "COM_DOUBLE_DRUM1_SPEED", "COM_DOUBLE_DRUM2_SPEED",
    "LUMPS_PRODUCTION_PD", "FINES_PRODUCTION_PD", "CHAR_PRODUCTION_PD",
    "IRON_ORE_CONSUMPTION_PD", "DOLOMITE_CONSUMPTION_PD",
    "FEED_COAL_CONSUMPTION_PD", "COARSE_COAL_CONSUMPTION_PD",
    "FINES_COAL_CONSUMPTION_PD", "COOLER_DISCHARGE_PD",
    "STEAM_PRODUCTION_PD", "PRODUCTION_FACTOR", "FINES_DAY_TOTAZIER",
    "PRODUTION_WET_SCRAPPER", "PRODUTION_LOW_MAG",
    "PRODUTION_COOLER_OVER_SIZE", "PRODUTION_CHAR_COAL",
    "LUMPS_DAY_TOTAZIER", "CHAR_DAY_TOTAZIER",
    "IRON_ORE_LOW_GRED_FACTOR", "IRON_ORE_HIGH_GRED_FACTOR",
    "ABC_GUN_CONTROL_VALVE1", "ABC_GUN_CONTROL_VALVE2",
    "ABC_GUN_CONTROL_VALVE3", "ABC_GUN_CONTROL_VALVE4",
    "ABC_GUN_CONTROL_VALVE5", "ABC_GUN_CONTROL_VALVE6",
    "ABC_GUN_CONTROL_VALVE7", "ABC_GUN_CONTROL_VALVE8",
    "ABC_GUN_CONTROL_VALVE9"
]

# Build SELECT list with table alias and quoted identifiers
columns_sql = ",\n    ".join([f't."{col}"' for col in COLUMNS])

# -------- SQL FOR DAILY DATA --------
sql_daily = f"""
SELECT
    {columns_sql}
FROM TSBSL.T_TSM_DRI_K1_PROCESS_DAILY t
WHERE t."TIMESTAMP" >= TO_TIMESTAMP(:1, 'YYYY-MM-DD HH24:MI')
  AND t."TIMESTAMP" <= TO_TIMESTAMP(:2, 'YYYY-MM-DD HH24:MI')
ORDER BY t."TIMESTAMP"
"""

# -------- EXECUTE AND BUILD FINAL DATASET --------
with oracledb.connect(user="imonitor", password="imtg", dsn=dsn) as conn:
    df_proc = pd.read_sql(sql_daily, con=conn, params=[START_DATE, END_DATE])

# Ensure TIMESTAMP is datetime and set as index
df_proc["TIMESTAMP"] = pd.to_datetime(df_proc["TIMESTAMP"])
df_proc = df_proc.set_index("TIMESTAMP").sort_index()

df_proc = df_proc.copy()
df_proc['SAMPLE_DATE'] = df_proc.index.date

# At this point df_daily is your final day-wise dataset
print("Daily dataframe shape:", df_proc.shape)
print(df_proc.head()) 

In [ ]:
import pandas as pd
import numpy as np

# --- 1. DATA PREPARATION: AGGREGATE PROCESS DATA DAILY ---

# We assume df_proc is the DataFrame with the full time index (hours/minutes)
# and df_coal is the DataFrame with the 'SP_COAL_CONSUMPTION' column.

# CRITICAL STEP: Reset the index, drop the time component, and group by date.
# We take the mean (average) of all process parameters for each day.
df_proc.reset_index(inplace=True) 

# Ensure the time column is named 'TIMESTAMP' and is datetime type
if 'TIMESTAMP' not in df_proc.columns:
    # If the time column is still the index, this will pull it out
    if df_proc.index.name != None:
        df_proc.reset_index(inplace=True)
        df_proc.rename(columns={df_proc.columns[0]: 'TIMESTAMP'}, inplace=True) # Assuming time is the first column after reset

df_proc['DATE_ONLY'] = df_proc['TIMESTAMP'].dt.date
df_proc_daily = df_proc.groupby('DATE_ONLY').mean(numeric_only=True)


# --- 2. PREPARE COAL DATA INDEX ---

# Create a 'DATE_ONLY' index for the coal data as well
df_coal.reset_index(inplace=True)
if 'TIMESTAMP' not in df_coal.columns:
    df_coal.rename(columns={df_coal.columns[0]: 'TIMESTAMP'}, inplace=True)

df_coal['DATE_ONLY'] = df_coal['TIMESTAMP'].dt.date
df_coal_daily = df_coal.set_index('DATE_ONLY')


# --- 3. MERGE THE DAILY DATAFRAMES ON INDEX ---

# Select only the consumption column from df_coal_daily (index is now DATE_ONLY)
df_coal_series = df_coal_daily['SP_COAL']

# Merge df_proc_daily with the coal data using the daily index.
df_merged = df_proc_daily.merge(df_coal_series, 
                              left_index=True, 
                              right_index=True, 
                              how='left')

# Check for zero rows or NaNs in the new merged data
print("--- Check on Merged Daily Data ---")
print(f"Total Merged Rows: {len(df_merged)}")
print(df_merged.head())
print(f"NaN Count in SP_COAL: {df_merged['SP_COAL'].isna().sum()}")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- 1. CONFIGURATION ---
# Target column (Quality)
target_col = 'SP_COAL'

# Assuming 'df_merged' is your complete DataFrame
# --- 2. DATA PREPARATION ---
# Select only numeric columns from the DataFrame
# This step is still important for filtering out text/category columns.
num_df = df_merged.select_dtypes(include='number')

# --- 3. CORRELATION CALCULATION AND SORTING ---
# KEY CHANGE: pandas corr() automatically excludes missing data (NaNs)
# in a pairwise fashion (meaning it only uses rows where BOTH variables are present).
# If a column is mostly or entirely NaN, the correlation against the target will be NaN.
# To ensure the correlation calculation runs cleanly:
# 1. We drop the target column itself for the series.
# 2. We sort the resulting series.
corr_series = num_df.corr(method='pearson')[target_col].drop(target_col)

# IMPORTANT FIX: Drop any resulting NaN correlations before sorting, 
# as these are caused by columns that are mostly or entirely NaN.
corr_series = corr_series.dropna() 

# Now sort the cleaned series from most positive (strongest positive) 
# to most negative (strongest negative) correlation.
corr_series_sorted = corr_series.sort_values(ascending=False)


# --- 4. SELECT TOP 10 ---
# The top 10 values in the sorted series are the 10 most positive correlations.
top10_pos = corr_series_sorted.head(10)

# The bottom 10 values in the sorted series are the 10 most negative correlations.
top10_neg = corr_series_sorted.tail(10)

# --- 5. PRINT RESULTS ---
print(f"===== Top 10 POSITIVE Correlations vs {target_col} =====")
print(top10_pos)

print(f"\n===== Top 10 NEGATIVE Correlations vs {target_col} =====")
print(top10_neg)

# --- 6. VISUALIZATION: Heatmap for Top 10 Positive ---
plt.figure(figsize=(7, 6))
sns.heatmap(top10_pos.to_frame(),
            annot=True,
            cmap='coolwarm', 
            center=0,
            linewidths=0.6,
            cbar=True,
            fmt=".3f") 
plt.title(f'Top 10 POSITIVE correlations vs {target_col}',
          fontsize=13, fontweight='bold')
plt.xlabel('Correlation Value')
plt.tight_layout()
plt.show()

# --- 7. VISUALIZATION: Heatmap for Top 10 Negative ---
plt.figure(figsize=(7, 6))
sns.heatmap(top10_neg.to_frame(),
            annot=True,
            cmap='coolwarm', 
            center=0,
            linewidths=0.6,
            cbar=True,
            fmt=".3f") 
plt.title(f'Top 10 NEGATIVE correlations vs {target_col}',
          fontsize=13, fontweight='bold')
plt.xlabel('Correlation Value')
plt.tight_layout()
plt.show()



Cell 5:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# --- 1. AUTOMATIC GLOBAL OUTLIER REMOVAL (IQR Method) ---
# This looks at every column and removes rows that are statistically 'extreme'
Q1 = df_merged.quantile(0.25)
Q3 = df_merged.quantile(0.75)
IQR = Q3 - Q1

# Define 'Normal' as anything within 1.5 * IQR of the middle 50%
# This removes startup/shutdown noise across ALL parameters automatically
df_final_clean = df_merged[~((df_merged < (Q1 - 1.5 * IQR)) | (df_merged > (Q3 + 1.5 * IQR))).any(axis=1)]

print(f"Global Cleaning complete.")
print(f"- Rows before: {len(df_merged)}")
print(f"- Rows after: {len(df_final_clean)}")

# --- 2. RECALCULATE TOP 10 (WHOLE DATASET) ---
target = 'SP_COAL'
numeric_data = df_final_clean.select_dtypes(include=[np.number])

# Calculate correlation for every single parameter vs the target
all_corrs = numeric_data.corr(method='pearson')[target].drop(target).dropna()

# Sort to find the global Top 10 Positive and Top 10 Negative
top10_pos = all_corrs.sort_values(ascending=False).head(10)
top10_neg = all_corrs.sort_values(ascending=True).head(10)

# --- 3. FINAL VISUALIZATION ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

sns.heatmap(top10_pos.to_frame(), annot=True, cmap='Reds', fmt=".3f", ax=ax1)
ax1.set_title(f'Global Top 10 POSITIVE vs {target}', fontweight='bold')

sns.heatmap(top10_neg.to_frame(), annot=True, cmap='Blues_r', fmt=".3f", ax=ax2)
ax2.set_title(f'Global Top 10 NEGATIVE vs {target}', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n===== TOP 10 NEGATIVE PARAMETERS =====")
print(top10_neg)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# --- 1. PREPARATION & CLEANING ---
target_col = 'SP_COAL'

# Remove non-process columns like 'index' or 'level_0' if they exist
cols_to_drop = ['index', 'level_0', 'DATE_ONLY']
df_working = df_merged.drop(columns=[c for c in cols_to_drop if c in df_merged.columns])

# Remove Outliers using the Interquartile Range (IQR) for the "most appropriate range"
Q1 = df_working.quantile(0.25)
Q3 = df_working.quantile(0.75)
IQR = Q3 - Q1
df_clean = df_working[~((df_working < (Q1 - 1.5 * IQR)) | (df_working > (Q3 + 1.5 * IQR))).any(axis=1)]

# --- 2. CALCULATE & SHORTLIST TOP 10 NEGATIVE ---
# Calculate Pearson correlation for all numeric parameters
corr_series = df_clean.select_dtypes(include='number').corr()[target_col].drop(target_col).dropna()

# Shortlist the top 10 strongest negative parameters
top10_neg_params = corr_series.sort_values(ascending=True).head(10).index.tolist()

# --- 3. VISUALIZATION (SCATTER PLOTS) ---
num_params = len(top10_neg_params)
cols = 2
rows = (num_params + 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 5))
axes = axes.flatten()

for i, param in enumerate(top10_neg_params):
    # sns.regplot plots the data points and the best-fit regression line
    sns.regplot(
        data=df_clean, 
        x=param, 
        y=target_col, 
        ax=axes[i],
        scatter_kws={'alpha':0.5, 's':15, 'color':'#2980b9'}, # Blue dots
        line_kws={'color':'#c0392b', 'lw':2}                  # Red regression line
    )
    
    # Calculate the specific correlation value for the title
    current_corr = corr_series[param]
    axes[i].set_title(f'{param}\nCorrelation: {current_corr:.3f}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(param, fontsize=10)
    axes[i].set_ylabel(target_col, fontsize=10)
    axes[i].grid(True, linestyle='--', alpha=0.6)

# Remove any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle(f'Top 10 Negative Parameters vs {target_col}\n(Outliers Removed - Daily Data)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print("\n===== TOP 10 NEGATIVE PARAMETERS SHORTLIST =====")
for idx, param in enumerate(top10_neg_params, 1):
    print(f"{idx}. {param}: {corr_series[param]:.4f}")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# --- 1. DEFINE THE PAIRS ---
# Ensure these names match your df_merged exactly (e.g., SAF_7_FLOW, ZONE_7_TEMP)
pairs = [
    ('SAF_7_SECONDARY_AIR', 'ZONE7_TEMP'),
    ('SAF_8_SECONDARY_AIR', 'ZONE8_TEMP'),
    ('SAF_5_SECONDARY_AIR', 'ZONE6_TEMP')
]

# --- 2. DATA CLEANING (Removing Outliers) ---
# We use IQR to focus on stable running hours
Q1 = df_merged.quantile(0.25)
Q3 = df_merged.quantile(0.75)
IQR = Q3 - Q1
df_clean = df_merged[~((df_merged < (Q1 - 1.5 * IQR)) | (df_merged > (Q3 + 1.5 * IQR))).any(axis=1)]

# --- 3. CALCULATION & PLOTTING ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

results = []

for i, (saf, zone) in enumerate(pairs):
    if saf in df_clean.columns and zone in df_clean.columns:
        # Calculate Pearson Correlation
        corr_val = df_clean[saf].corr(df_clean[zone])
        results.append({'Pair': f'{saf} vs {zone}', 'Correlation': corr_val})
        
        # Plot Scatter + Regression Line
        sns.regplot(data=df_clean, x=saf, y=zone, ax=axes[i], 
                    scatter_kws={'alpha':0.5}, line_kws={'color':'red'})
        
        axes[i].set_title(f'{saf} vs {zone}\nCorr: {corr_val:.3f}', fontweight='bold')
        axes[i].grid(True, alpha=0.3)
    else:
        axes[i].text(0.5, 0.5, f'Columns Not Found:\n{saf} or {zone}', 
                     ha='center', va='center')

plt.tight_layout()
plt.show()

# --- 4. SUMMARY TABLE ---
print("--- Correlation Results: Cause (SAF) vs Effect (Zone Temp) ---")
print(pd.DataFrame(results))

In [ ]:
import pandas as pd
import numpy as np

# --- 1. DATA PREPARATION ---
# Target columns and their respective zone effects
saf_params = ['SAF_5_SECONDARY_AIR', 'SAF_7_SECONDARY_AIR', 'SAF_8_SECONDARY_AIR']
zone_params = ['ZONE_6_TEMP', 'ZONE_7_TEMP', 'ZONE_8_TEMP']
target_col = 'SP_COAL'

# Apply IQR cleaning to focus on steady running hours
df_clean = df_merged.copy()
for col in saf_params + zone_params + [target_col]:
    if col in df_clean.columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        df_clean = df_clean[(df_clean[col] >= (Q1 - 1.5 * IQR)) & (df_clean[col] <= (Q3 + 1.5 * IQR))]

# --- 2. DEFINE PEAK EFFICIENCY ---
# We look at days where coal consumption was in the bottom 10% (Best Performance)
best_perf_threshold = df_clean[target_col].quantile(0.10)
df_best = df_clean[df_clean[target_col] <= best_perf_threshold]

# --- 3. CALCULATE OPTIMUM RANGES ---
optimum_results = []

for saf in saf_params:
    if saf in df_best.columns:
        optimum_results.append({
            'SAF Fan': saf,
            'Current Avg': f"{df_clean[saf].mean():.2f}",
            'Optimum Avg (Target)': f"{df_best[saf].mean():.2f}",
            'Optimum Min': f"{df_best[saf].min():.2f}",
            'Optimum Max': f"{df_best[saf].max():.2f}",
            'Potential Increase': f"{df_best[saf].mean() - df_clean[saf].mean():.2f}"
        })

# --- 4. OUTPUT REPORT ---
df_optimum = pd.DataFrame(optimum_results)
print(f"--- KILN OPTIMIZATION REPORT: SAF SETPOINTS ---")
print(f"Goal: Minimize {target_col} (Current Target: <= {best_perf_threshold:.2f})")
print("-" * 85)
print(df_optimum.to_string(index=False))

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# --- 1. DATA CLEANING ---
target_col = 'SP_COAL'

# Explicitly remove the 'index' and other non-process labels
df_working = df_merged.drop(columns=[c for c in ['index', 'level_0', 'DATE_ONLY'] if c in df_merged.columns])

# IQR Filter to remove outliers and focus on "Stable Kiln" data
Q1 = df_working.quantile(0.25)
Q3 = df_working.quantile(0.75)
IQR = Q3 - Q1
df_clean = df_working[~((df_working < (Q1 - 1.5 * IQR)) | (df_working > (Q3 + 1.5 * IQR))).any(axis=1)]

# --- 2. IDENTIFY TOP 10 POSITIVE ---
corr_series = df_clean.select_dtypes(include='number').corr()[target_col].drop(target_col).dropna()
top10_pos_params = corr_series.sort_values(ascending=False).head(10).index.tolist()

# --- 3. LARGE INDIVIDUAL PLOTTING ---
# We use a vertical stack (10 rows, 1 column) with a very large height
fig, axes = plt.subplots(nrows=10, ncols=1, figsize=(12, 60)) 

for i, param in enumerate(top10_pos_params):
    sns.regplot(
        data=df_clean, 
        x=param, 
        y=target_col, 
        ax=axes[i],
        scatter_kws={'alpha':0.5, 's':30, 'color':'#d35400'}, # Larger points
        line_kws={'color':'#2c3e50', 'lw':3} # Thicker line
    )
    
    # Large titles and labels for readability
    axes[i].set_title(f'Parameter: {param} | Correlation: {corr_series[param]:.3f}', 
                      fontsize=16, fontweight='bold', pad=15)
    axes[i].set_xlabel(f'{param} Value', fontsize=12)
    axes[i].set_ylabel(target_col, fontsize=12)
    axes[i].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout(pad=5.0) # Adds space between plots so they don't overlap

# This will save a very long image file that contains all plots at full size
plt.savefig('Full_Size_Positive_Plots.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd

# Load your dataset
# df = pd.read_csv('your_data_file.csv') 

# 1. Correlation: Secondary Air vs. Zone 11 Temperature
corr_air_temp = df_working['SAF_2__SECONDARY_AIR'].corr(df_working['ZONE11_TEMP'])

# 2. Correlation: Secondary Air vs. Kiln Inlet Pressure
corr_air_pressure = df_working['SAF_2__SECONDARY_AIR'].corr(df_working['KILN_INLET_PRESSURE'])

print(f"Correlation (Secondary Air & Zone 11 Temp): {corr_air_temp:.4f}")
print(f"Correlation (Secondary Air & Inlet Pressure): {corr_air_pressure:.4f}")

# Optional: View them in a small matrix format
subset_corr = df_working[['SAF_2__SECONDARY_AIR', 'ZONE11_TEMP', 'KILN_INLET_PRESSURE']].corr()
print("\nCorrelation Matrix:")
print(subset_corr)

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Using the dataframe name 'df_working' from your screenshot
# Setting the style to match your dark/grid background
plt.figure(figsize=(8, 6))
sns.set_style("whitegrid")

# Create the regression plot
ax = sns.regplot(
    data=df_working, 
    x='SAF_2__SECONDARY_AIR', 
    y='KILN_INLET_PRESSURE',
    scatter_kws={'color': 'steelblue', 'alpha': 0.6, 's': 20},
    line_kws={'color': 'red', 'lw': 2}
)

# Calculate correlation for the title
correlation = df_working['SAF_2__SECONDARY_AIR'].corr(df_working['KILN_INLET_PRESSURE'])

# Formatting the plot to match your existing style
plt.title(f'SAF_2_SECONDARY_AIR vs KILN_INLET_PRESSURE\nCorr: {correlation:.3f}', fontsize=12)
plt.xlabel('SAF_2__SECONDARY_AIR')
plt.ylabel('KILN_INLET_PRESSURE')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Using your existing dataframe name from the screenshots
# df_working = pd.read_csv('your_data.csv') 

# Define the target variable
target = 'KMD_RPM'

# Define the other parameters to check against
others = ['ZONE11_TEMP', 'SAF_2__SECONDARY_AIR', 'KILN_INLET_PRESSURE']

print(f"--- Correlation Results for {target} ---")

for param in others:
    correlation_value = df_working[target].corr(df_working[param])
    print(f"{target} vs {param}: {correlation_value:.4f}")

# Displaying as a dedicated correlation row for easy reading
kmd_corr_row = df_working[[target] + others].corr().loc[[target]]
print("\nCorrelation Matrix Row:")
print(kmd_corr_row)



import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Using the dataframe name 'df_working' from your screenshot
# Setting the style to match your dark/grid background
plt.figure(figsize=(8, 6))
sns.set_style("whitegrid")

# Create the regression plot
ax = sns.regplot(
    data=df_working, 
    x='KMD_RPM', 
    y='KILN_INLET_PRESSURE',
    scatter_kws={'color': 'steelblue', 'alpha': 0.6, 's': 20},
    line_kws={'color': 'red', 'lw': 2}
)

# Calculate correlation for the title
correlation = df_working['KMD_RPM'].corr(df_working['KILN_INLET_PRESSURE'])

# Formatting the plot to match your existing style
plt.title(f'KMD_RPM vs KILN_INLET_PRESSURE\nCorr: {correlation:.3f}', fontsize=12)
plt.xlabel('KMD_RPM')
plt.ylabel('KILN_INLET_PRESSURE')

plt.tight_layout()
plt.show()